In [ ]:
import torch
if torch.cuda.is_available():
    print("gpu available")
    print(torch.cuda.get_device_name(0))
else:
    print("no gpu available")

gpu available
NVIDIA A100-SXM4-80GB


In [ ]:
!pip install wfdb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 137.8 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.1 which is incompatible.
dask-cudf-cu12 25.10.0 requires pandas<2.4.0dev0,>=2.0, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but

In [ ]:
import wfdb #load ecg data from physionet
import numpy as np # numerical operations
import matplotlib.pyplot as plt # plotting
import torch # build model
import torch.nn as nn # neural network, defines layers and structure of model
import torch.optim as optim # optimizers used to adjust model weights
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader # batch data during training + eval
from sklearn.model_selection import train_test_split # to split and evaluate data
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from google.colab import drive
import os
from datetime import date, datetime
from zoneinfo import ZoneInfo
import pickle

In [ ]:
drive.mount('/content/drive')

# change this based on where your ecg_data.npz is!
save_path = '/content/drive/MyDrive/ZiyaAhmad/12sec_ecg_data.npz'
data = np.load(save_path)
all_windows_original = data['windows']
all_labels_3class = data['labels']

# change this based on where your train_test_split.npz is!
split_path = '/content/drive/MyDrive/ZiyaAhmad/12sec_train_test_split.npz'
split_data = np.load(split_path)
train_indices_all = split_data['train_indices']
test_indices_all = split_data['test_indices']

abnormal_mask = (all_labels_3class == 1) | (all_labels_3class == 2)
abnormal_indices = np.where(abnormal_mask)[0]

print("loaded pre-saved train/test split samples")
print(f"total train samples: {len(train_indices_all):,}")
print(f"total test samples: {len(test_indices_all)}")

print(f"\ntotal abnormal samples: {len(abnormal_indices)}")

# get abnromal samples in train/test
train_indices = np.intersect1d(train_indices_all, abnormal_indices)
test_indices = np.intersect1d(test_indices_all, abnormal_indices)

print(f"\nfiltered abnormal only:")
print(f"train samples (sva + presva): {len(train_indices):,}")
print(f"test samples (sva + presva): {len(test_indices)}")

#extract only the abnormal samples
all_windows = all_windows_original[abnormal_indices]
all_labels_3class_filtered = all_labels_3class[abnormal_indices]

# convert labels: 1(sva) -> 0, 2 (presva) -> 1
all_labels_binary = all_labels_3class_filtered - 1

# map from the global indices to local indices
# create a mapping from global index to local idnex
global_to_local = {global_idx: local_idx for local_idx, global_idx in enumerate(abnormal_indices)}

#map train/test indices to local indices
train_indices_local = np.array([global_to_local[idx] for idx in train_indices])
test_indices_local = np.array([global_to_local[idx] for idx in test_indices])

Mounted at /content/drive
loaded pre-saved train/test split samples
total train samples: 197,044
total test samples: 49262

total abnormal samples: 100813

filtered abnormal only:
train samples (sva + presva): 80,650
test samples (sva + presva): 20163


In [ ]:
train_labels = all_labels_binary[train_indices_local]

sva_count = np.sum(train_labels == 0)
presva_count = np.sum(train_labels == 1)

print(f"current training set class distribution")
print(f"  SVA (0): {sva_count:,}")
print(f"  Pre-SVA (1): {presva_count:,}")
print(f"  Total: {len(train_labels):,}")
print(f"  Ratio SVA:Pre-SVA = {sva_count/presva_count:.2f}:1")

current training set class distribution
  SVA (0): 62,565
  Pre-SVA (1): 18,085
  Total: 80,650
  Ratio SVA:Pre-SVA = 3.46:1


In [ ]:
def augment_presva(windows, labels, aug_factor=2):
    # augment pre-sva w/ realistic ecg variations
    # args:
    #     windows: ecg signal windows (n, 1024)
    #     labels: binary labels (0=sva, 1=pre-sva)
    #     aug_factor: how many augmented versions per pre-sva sample
    # returns:
    #     augmented windows & labels
    augmented_windows = []
    augmented_labels = []

    for i in range(len(labels)):
        augmented_windows.append(windows[i]) #always inc. orig ecg
        augmented_labels.append(labels[i])

        if labels[i] == 1: #only augment pre-sva
            original = windows[i]

            for _ in range(aug_factor):
                aug_type = np.random.choice(['noise','scale','shift','combo'])

                if aug_type == 'noise':
                    noise_level = np.random.uniform(0.005, 0.02)
                    augmented = original + np.random.normal(0,noise_level,original.shape)

                elif aug_type == 'scale':
                    scale_factor = np.random.uniform(0.90,1.10)
                    augmented = original*scale_factor

                elif aug_type == 'shift':
                    shift_amount = np.random.randint(-15,15)
                    augmented = np.roll(original,shift_amount)

                elif aug_type == 'combo':
                    noise_level = np.random.uniform(0.005,0.015)
                    scale_factor = np.random.uniform(0.92,1.08)
                    augmented = (original + np.random.normal(0,noise_level,original.shape)) * scale_factor

                augmented_windows.append(augmented)
                augmented_labels.append(1)

    return np.array(augmented_windows), np.array(augmented_labels)


# apply augmentation to TRAINING data only
print(f"Before augmentation:")
print(f"  Train SVA: {np.sum(all_labels_binary[train_indices_local] == 0):,}")
print(f"  Train Pre-SVA: {np.sum(all_labels_binary[train_indices_local] == 1):,}")

# get training data
train_windows = all_windows[train_indices_local]
train_labels = all_labels_binary[train_indices_local]

# augment
train_windows_aug, train_labels_aug = augment_presva(train_windows, train_labels, aug_factor=3)

print(f"\nAfter augmentation:")
print(f"  Train SVA: {np.sum(train_labels_aug == 0):,}")
print(f"  Train Pre-SVA: {np.sum(train_labels_aug == 1):,}")

Before augmentation:
  Train SVA: 62,565
  Train Pre-SVA: 18,085

After augmentation:
  Train SVA: 62,565
  Train Pre-SVA: 72,340


In [ ]:
# normalize the data
scaler = StandardScaler()
all_windows_flat = all_windows.reshape(-1, 1)
all_windows_scaled = scaler.fit_transform(all_windows_flat).reshape(all_windows.shape)

# convert to tensors
X = torch.tensor(all_windows_scaled, dtype=torch.float32).unsqueeze(-1)
y = torch.tensor(all_labels_binary, dtype=torch.float32)

# use the pre-defined split indices instead of train_test_split()
X_train = X[train_indices_local]
X_test = X[test_indices_local]
y_train = y[train_indices_local]
y_test = y[test_indices_local]

print(f"Final dataset sizes:")
print(f"Training samples: {len(X_train):,}")
print(f"Test samples: {len(X_test):,}")

print(f"\nTest set breakdown:")
print(f"  SVA (0): {torch.sum(y_test == 0).item():,}")
print(f"  Pre-SVA (1): {torch.sum(y_test == 1).item():,}")

# create dataloaders
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

Final dataset sizes:
Training samples: 80,650
Test samples: 20,163

Test set breakdown:
  SVA (0): 15,641
  Pre-SVA (1): 4,522


In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(SelfAttention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)
        self.layer_norm = nn.LayerNorm(hidden_dim)

    def forward(self, gru_output):
        gru_output = self.layer_norm(gru_output)
        attn_scores = self.attention(gru_output)
        attn_weights = F.softmax(attn_scores, dim=1)
        context = torch.sum(attn_weights * gru_output, dim=1)
        return context, attn_weights


class GRUWithAttention(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=2, dropout=0.3):
        super(GRUWithAttention, self).__init__()

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.attention = SelfAttention(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        gru_out, _ = self.gru(x)
        attended, attn_weights = self.attention(gru_out)
        attended = self.dropout(attended)
        out = self.fc(attended)
        return self.sigmoid(out)


# training setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GRUWithAttention(dropout=0.3).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0002)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

Training samples: 80650
Test samples: 20163


In [ ]:
num_epochs = 300

train_losses = []
test_losses = []
test_accuracies = []
macro_avgs = []

best_macro_avg = 0
best_accuracy = 0
best_model_state = None
patience = 0
max_patience = 20

for epoch in range(num_epochs):
    model.train() # put model in traning mode
    epoch_loss = 0 # accumulate loss for this epoch
    train_correct = 0
    train_total = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device) # move to gpu if available
        y_batch = y_batch.to(device).unsqueeze(1) # reshape: (32,) -> (32,1)

        outputs = model(X_batch) # forward pass, get model predictions

        loss = criterion(outputs, y_batch) # calculate loss

        # backward pass: calc gradients
        optimizer.zero_grad() #clear old gradients
        loss.backward() # calculate new gradients
        optimizer.step() # update model weights

        epoch_loss += loss.item() # track total loss for this epoch

        # calc training accuracy
        predicted = (outputs > 0.5).float()
        train_correct += (predicted == y_batch).sum().item()
        train_total += y_batch.size(0)

    avg_train_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    train_acc = train_correct / train_total

    model.eval() # put model in evaluation mode
    test_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device) # y_batch here is (batch_size,)

            outputs_model = model(X_batch) # outputs_model is (batch_size, 1)

            #convert probabilities to predictions
            # if output > 0.5 -> predict 1 (abnormal)
            # if output <= 0.5 -> predict 0 (normal)
            # Use .view(-1) to ensure predicted is always a 1D tensor
            predicted = (outputs_model.view(-1) > 0.5).float()

            # calc loss
            loss = criterion(outputs_model, y_batch.unsqueeze(1)) # Use outputs_model directly for loss
            test_loss += loss.item()

            #calc accuracy
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)

            # collect for f1 score
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

        avg_test_loss = test_loss / len(test_loader)
        test_losses.append(avg_test_loss)

        accuracy = correct / total
        test_accuracies.append(accuracy)

        macro_avg = f1_score(all_labels, all_preds, average='macro')
        macro_avgs.append(macro_avg)

        print(f"Epoch {epoch+1}/{num_epochs} - Val Loss: {avg_test_loss:.4f} | Val Acc: {accuracy:.4f} | Macro Avg: {macro_avg:.4f} | Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f}", end="")

        if macro_avg > best_macro_avg:
            best_macro_avg = macro_avg
            best_model_state = model.state_dict().copy()
            patience = 0
            print("\n")
        else:
            patience += 1
            print(f" [{patience}/{max_patience}]\n")

        # if accuracy > best_accuracy:
        #     best_accuracy = accuracy
        #     best_model_state = model.state_dict().copy()

        if patience >= max_patience:
            break

        if (epoch + 1) % 5 == 0:
            print(classification_report(all_labels, all_preds, target_names=['SVA', 'Pre-SVA'], digits=4))

print(f"Best Macro-Average: {best_macro_avg:.4f}")
model.load_state_dict(best_model_state)

Epoch 1/300 - Val Loss: 0.5265 | Val Acc: 0.7757 | Macro Avg: 0.4369 | Train Loss: 0.5353 | Train Acc: 0.7757

Epoch 2/300 - Val Loss: 0.5223 | Val Acc: 0.7757 | Macro Avg: 0.4369 | Train Loss: 0.5276 | Train Acc: 0.7757 [1/20]

Epoch 3/300 - Val Loss: 0.5196 | Val Acc: 0.7757 | Macro Avg: 0.4369 | Train Loss: 0.5240 | Train Acc: 0.7756 [2/20]

Epoch 4/300 - Val Loss: 0.5165 | Val Acc: 0.7757 | Macro Avg: 0.4369 | Train Loss: 0.5209 | Train Acc: 0.7759 [3/20]

Epoch 5/300 - Val Loss: 0.5016 | Val Acc: 0.7774 | Macro Avg: 0.4460 | Train Loss: 0.5125 | Train Acc: 0.7760

              precision    recall  f1-score   support

         SVA     0.7772    0.9996    0.8745     15641
     Pre-SVA     0.8696    0.0088    0.0175      4522

    accuracy                         0.7774     20163
   macro avg     0.8234    0.5042    0.4460     20163
weighted avg     0.7979    0.7774    0.6823     20163

Epoch 6/300 - Val Loss: 0.4793 | Val Acc: 0.7890 | Macro Avg: 0.5659 | Train Loss: 0.4898 | Train

<All keys matched successfully>

In [ ]:
# if you manually stopped the last kernel, uncomment the next line to load your best model
# model.load_state_dict(best_model_state)

#alter based on where you want to save your model
save_path = '/content/drive/MyDrive/'
os.makedirs(save_path, exist_ok=True)

# alter based on your time zone
pst_tz = ZoneInfo("America/Los_Angeles")
today = date.today()
now = datetime.now(tz = pst_tz)
formatted_time = now.strftime("%H:%M")

specific_path = f"{best_macro_avg:.4f}|{str(formatted_time)}|{str(today)}|"
torch.save(model.state_dict(), save_path + specific_path)

training_info = {
    'best_accuracy' : best_accuracy,
    'best_macro_avg' : best_macro_avg,
    'train_losses' : train_losses,
    'test_losses' : test_losses,
    'test_accuracies' : test_accuracies,
    'final_epoch' : len(train_losses)
}

with open(save_path + specific_path + '.pkl', 'wb') as f:
    pickle.dump(training_info, f)

print("model saved to google drive")
print(f"location: {save_path}")
print(f"model file: {specific_path}")
print(f"best macro-avg: {best_macro_avg:.4f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
model saved to google drive
location: /content/drive/MyDrive/ZiyaAhmad/science/Synopsys 2025-2026/Code/best models/sva_vs_presva/
model file: 0.9717|15:08|2026-02-18|
best macro-avg: 0.9717
